# EEG_18 — Functional Clustering dei Soggetti

**Domanda**: perché alcuni soggetti si decodificano meglio di altri?

EEG_16 ha trovato due cluster strutturali (connettività locale vs diffusa), ma questi cluster
**non** predicono la bAcc (p=0.8). La struttura del grafo descrive *il tipo di cervello*,
non *quanto bene risponde al task*.

Questo notebook cerca la risposta nel comportamento **funzionale** del segnale durante il task:
- **Within-class consistency**: il cervello riproduce lo stesso pattern EEG per la stessa parola?
- **Fisher ratio**: le classi sono separabili nel segnale grezzo?
- **Alpha power**: quanto è impegnato il soggetto durante l'imagery?
- **Trial variance**: SNR generale del segnale

**Pipeline**:
1. Feature funzionali per soggetto (da segnale raw `d['x']` dei .pt files)
2. Clustering su queste feature → gruppi funzionali
3. Post-hoc: cluster vs bAcc (test ipotesi)
4. Quale feature funzionale correla con bAcc?
5. ARI vs EEG_16: funzionale ≠ strutturale?

## §1 — Setup & Config

In [ ]:
import json, logging, re
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.signal import welch
from scipy.stats import mannwhitneyu, pearsonr
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)-8s %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('eeg18')

project_root = next((p for p in [Path.cwd()] + list(Path.cwd().parents)
                     if (p / '.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
CKPT_DIR = project_root / 'models' / 'eeg18'; CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── config ─────────────────────────────────────────────────────────────────────
METRIC      = 'abs_pcc'
N_CHAN      = 61
N_SAMPLES   = 384
FS          = 256
K_RANGE     = [2, 3, 4]
RANDOM_SEED = 42

ACC_FILE_12 = project_root / 'figures' / 'eeg12_subject_ranking.csv'
ACC_FILE_13 = project_root / 'figures' / 'eeg13_subject_ranking.csv'

label2cluster = {int(k): int(v) for k, v in json.loads(
    (project_root / 'configs' / 'label_schemes' / 'labelid2cluster_concr4.json').read_text()).items()}
word2label = {k: int(v) for k, v in json.loads(
    (project_root / 'configs' / 'label_schemes' / 'label2idx.json').read_text()).items()}

log.info(f'project_root = {project_root}')
log.info(f'Metrica: {METRIC} | FS: {FS} Hz | N_CHAN: {N_CHAN} | N_SAMPLES: {N_SAMPLES}')

## §2 — Index & Calcolo Feature Funzionali per Soggetto

In [ ]:
# ── DIAGNOSTICA: ispeziona il primo .pt trovato ───────────────────────────────
_PAT_diag = re.compile(r'^P(\d+)_S(\d+)$')
HG_ROOT_diag = project_root / 'data' / f'hypergraphs_pruned_{METRIC}'
log.info(f'HG_ROOT: {HG_ROOT_diag}  exists={HG_ROOT_diag.exists()}')

_first_pts = list(HG_ROOT_diag.rglob('trial_*.pt'))[:3]
log.info(f'Primi .pt trovati: {len(_first_pts)}')
for _p in _first_pts:
    try:
        _d = torch.load(_p, weights_only=False)
        log.info(f'  {_p.name} — keys={list(_d.keys())}')
        for k, v in _d.items():
            if hasattr(v, 'shape'):
                log.info(f'    {k}: shape={v.shape} dtype={v.dtype}')
            else:
                log.info(f'    {k}: {v}')
    except Exception as e:
        log.error(f'  {_p.name} — ERRORE: {e}')

In [ ]:
# ── DIAGNOSTICA 2: primo soggetto, prime 5 path — senza try/except ───────────
_PAT2 = re.compile(r'^P(\d+)_S(\d+)$')
_HG   = project_root / 'data' / f'hypergraphs_pruned_{METRIC}'
_idx2 = defaultdict(list)
for p in sorted(_HG.rglob('trial_*.pt')):
    m = _PAT2.match(p.parent.name)
    if m:
        _idx2[int(m.group(1))].append(p)

_sid0   = sorted(_idx2.keys())[0]
_paths0 = _idx2[_sid0][:5]
log.info(f'Test su soggetto {_sid0}, {len(_paths0)} trial')

for _p in _paths0:
    _d = torch.load(_p, weights_only=False)
    _x = _d['x'].float().numpy()
    _y_raw  = _d['y']
    _y_word = int(_y_raw.squeeze()) if isinstance(_y_raw, torch.Tensor) else int(_y_raw)
    _lbl    = label2cluster.get(_y_word)
    log.info(f'  {_p.name}  x.shape={_x.shape}  y={_y_word}  cluster={_lbl}  '
             f'word={_d.get("meta", {}).get("word", "?")}')

In [ ]:
_PAT  = re.compile(r'^P(\d+)_S(\d+)$')
HG_ROOT = project_root / 'data' / f'hypergraphs_pruned_{METRIC}'

# Indice: soggetto → lista path .pt
IDX = defaultdict(list)
for p in sorted(HG_ROOT.rglob('trial_*.pt')):
    m = _PAT.match(p.parent.name)
    if m:
        IDX[int(m.group(1))].append(p)

ALL_SUBJ = sorted(IDX.keys())
log.info(f'Soggetti trovati: {len(ALL_SUBJ)}')


def compute_functional_features(paths, min_trials=20):
    """
    Carica i trial raw (d['x']) per un soggetto e calcola 4 feature funzionali.
    Usa d['meta']['word'] per il cluster label (più robusto di d['y']).
    Ritorna dict o None se dati insufficienti.
    """
    X_list, y_list = [], []
    skip_shape, skip_label, skip_err = 0, 0, 0

    for p in paths:
        try:
            d = torch.load(p, weights_only=False)
            x = d['x'].float().numpy()
            if x.shape != (N_CHAN, N_SAMPLES):
                skip_shape += 1
                continue
            # cluster label via meta['word'] → word2label → label2cluster
            word = d.get('meta', {}).get('word', None)
            if word is not None:
                idx_w = word2label.get(word)
                lbl   = label2cluster.get(idx_w) if idx_w is not None else None
            else:
                y_raw  = d['y']
                y_word = int(y_raw.squeeze()) if isinstance(y_raw, torch.Tensor) else int(y_raw)
                lbl    = label2cluster.get(y_word)
            if lbl is None:
                skip_label += 1
                continue
            X_list.append(x)
            y_list.append(lbl)
        except Exception as e:
            skip_err += 1
            continue

    total = len(paths)
    log.debug(f'  trials: ok={len(X_list)}/{total}  skip_shape={skip_shape}  '
              f'skip_label={skip_label}  skip_err={skip_err}')

    if len(X_list) < min_trials:
        return None

    X = np.stack(X_list)
    y = np.array(y_list)
    classes = np.unique(y)

    # ── 1. Within-class consistency ──────────────────────────────────────────
    consistency_vals = []
    for c in classes:
        idx = np.where(y == c)[0]
        if len(idx) < 2:
            continue
        proto = X[idx].mean(axis=0)
        for i in idx:
            r = np.corrcoef(X[i].ravel(), proto.ravel())[0, 1]
            if np.isfinite(r):
                consistency_vals.append(r)
    within_consistency = float(np.mean(consistency_vals)) if consistency_vals else np.nan

    # ── 2. Fisher ratio ──────────────────────────────────────────────────────
    X_feat     = X.mean(axis=2)
    grand_mean = X_feat.mean(axis=0)
    class_means = np.array([X_feat[y == c].mean(axis=0) for c in classes])
    between_var = np.mean([np.sum((m - grand_mean) ** 2) for m in class_means])
    within_var  = np.mean([X_feat[y == c].var(axis=0).mean() for c in classes])
    fisher_ratio = float(between_var / (within_var + 1e-9))

    # ── 3. Alpha power (8–12 Hz) ─────────────────────────────────────────────
    X_2d = X.reshape(-1, N_SAMPLES)
    freqs, psd = welch(X_2d, fs=FS, nperseg=128, axis=1)
    alpha_mask  = (freqs >= 8) & (freqs <= 12)
    alpha_power = float(psd[:, alpha_mask].mean())

    # ── 4. Trial variance ────────────────────────────────────────────────────
    trial_variance = float(X.var(axis=(1, 2)).mean())

    return {
        'within_consistency': within_consistency,
        'fisher_ratio':       fisher_ratio,
        'alpha_power':        alpha_power,
        'trial_variance':     trial_variance,
        'n_trials':           len(X_list),
    }


FEAT_CACHE = CKPT_DIR / 'feat_functional.npz'
FEAT_NAMES = ['within_consistency', 'fisher_ratio', 'alpha_power', 'trial_variance']

def _load_cache(path):
    """Carica cache solo se valida (almeno 1 soggetto)."""
    if not path.exists():
        return None
    _c = np.load(path)
    if len(_c['subj_ids']) == 0:
        log.warning('Cache vuota trovata — eliminazione e ricalcolo')
        path.unlink()
        return None
    return _c

_cache = _load_cache(FEAT_CACHE)
if _cache is not None:
    log.info('Cache valida trovata — caricamento veloce')
    FEAT_F     = _cache['feats']
    SUBJ_F     = _cache['subj_ids'].tolist()
    N_TRIALS_F = _cache['n_trials'].tolist()
else:
    feats_list, subj_list, ntrials_list = [], [], []
    for sid in tqdm(ALL_SUBJ, desc='Soggetti'):
        res = compute_functional_features(IDX[sid])
        if res is None:
            log.warning(f'Soggetto {sid}: dati insufficienti, saltato')
            continue
        feats_list.append([res[k] for k in FEAT_NAMES])
        subj_list.append(sid)
        ntrials_list.append(res['n_trials'])

    if not feats_list:
        raise RuntimeError('Nessun soggetto valido — controlla HG_ROOT e label2cluster')

    FEAT_F     = np.array(feats_list, dtype=np.float32)
    SUBJ_F     = subj_list
    N_TRIALS_F = ntrials_list
    np.savez(FEAT_CACHE, feats=FEAT_F, subj_ids=np.array(SUBJ_F),
             n_trials=np.array(N_TRIALS_F))
    log.info('Cache salvata')

log.info(f'Feature funzionali: {FEAT_F.shape} — {len(SUBJ_F)} soggetti')
log.info(f'Trial per soggetto: min={min(N_TRIALS_F)}  max={max(N_TRIALS_F)}  '
         f'mean={np.mean(N_TRIALS_F):.0f}')

## §3 — Distribuzione Feature Funzionali

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Distribuzione feature funzionali per soggetto', fontsize=13, fontweight='bold')

labels_nice = ['Within-class\nconsistency', 'Fisher\nratio', 'Alpha power\n(8–12 Hz)', 'Trial\nvariance']

for ax, feat_col, name in zip(axes, FEAT_F.T, labels_nice):
    ax.hist(feat_col, bins=15, color='steelblue', edgecolor='white', linewidth=0.5)
    ax.axvline(np.median(feat_col), color='tomato', lw=2, label=f'median={np.median(feat_col):.3f}')
    ax.set_title(name, fontsize=11)
    ax.legend(fontsize=8)
    ax.set_xlabel('Valore', fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(FIG_DIR / 'eeg18_feat_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## §4 — Clustering Funzionale (KMeans, k=2–4)

In [ ]:
scaler = StandardScaler()
n_comp = min(len(SUBJ_F) - 1, FEAT_F.shape[1], 20)
pca    = PCA(n_components=n_comp, random_state=RANDOM_SEED)

X_sc  = scaler.fit_transform(FEAT_F)
X_pca = pca.fit_transform(X_sc)

log.info(f'PCA n_components={n_comp}  varianza spiegata: {pca.explained_variance_ratio_.round(3)}')

sil_scores = {}
all_labels = {}
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=20)
    lbl = km.fit_predict(X_pca)
    sil = silhouette_score(X_pca, lbl)
    sil_scores[k] = sil
    all_labels[k] = lbl
    log.info(f'k={k}  silhouette={sil:.4f}  counts={np.bincount(lbl).tolist()}')

k_best = max(sil_scores, key=sil_scores.get)
LABELS_F = all_labels[k_best]
log.info(f'\nBest k={k_best}  sil={sil_scores[k_best]:.4f}')

PALETTE_F = ['#4878CF', '#D65F5F', '#6ACC65', '#B47CC7'][:k_best]

# Plot silhouette bar + PCA 2D
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Clustering funzionale — selezione k', fontsize=13, fontweight='bold')

ax = axes[0]
ax.bar(list(sil_scores.keys()), list(sil_scores.values()),
       color=['#4878CF' if k != k_best else '#D65F5F' for k in K_RANGE])
ax.set_xlabel('k'); ax.set_ylabel('Silhouette score')
ax.set_title('Silhouette per k')
for k, s in sil_scores.items():
    ax.text(k, s + 0.002, f'{s:.3f}', ha='center', fontsize=10)

ax = axes[1]
pca2 = PCA(n_components=2, random_state=RANDOM_SEED)
Z2   = pca2.fit_transform(X_sc)
for c in range(k_best):
    mask = LABELS_F == c
    ax.scatter(Z2[mask, 0], Z2[mask, 1], c=PALETTE_F[c],
               label=f'Cluster {c} (n={mask.sum()})', s=60, alpha=0.8,
               edgecolors='white', lw=0.5)
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'PCA 2D — k={k_best} (feature funzionali)')
ax.legend(fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(FIG_DIR / 'eeg18_clustering_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

## §5 — Caricamento bAcc (EEG_12 / EEG_13)

In [ ]:
acc_data = {}
for label, fpath in [('EEG_12', ACC_FILE_12), ('EEG_13', ACC_FILE_13)]:
    if fpath.exists():
        df = pd.read_csv(fpath)
        if 'Subject' in df.columns:
            df['sid'] = df['Subject'].str.extract(r'(\d+)').astype(int)
        col = 'Test bAcc' if 'Test bAcc' in df.columns else 'test_bacc'
        acc_data[label] = dict(zip(df['sid'], df[col]))
        log.info(f'{label}: {len(acc_data[label])} soggetti')
    else:
        log.warning(f'{label} non trovato: {fpath}')

acc12 = acc_data.get('EEG_12', {})
acc13 = acc_data.get('EEG_13', {})
ACC   = acc13 if acc13 else acc12
log.info(f'Accuracy primaria: {"EEG_13" if acc13 else "EEG_12"} ({len(ACC)} soggetti)')

## §6 — Post-hoc: Cluster Funzionali vs bAcc

In [ ]:
# allinea bAcc con SUBJ_F
bacc_f = np.array([ACC.get(sid, np.nan) for sid in SUBJ_F])
valid  = ~np.isnan(bacc_f)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Feature Funzionali — cluster vs bAcc (k={k_best})', fontsize=13, fontweight='bold')

# ── Boxplot ──────────────────────────────────────────────────────────────────
ax = axes[0]
data_per_cluster = [bacc_f[(LABELS_F == c) & valid] for c in range(k_best)]
bp = ax.boxplot(data_per_cluster, patch_artist=True, widths=0.5,
                medianprops={'color': 'orange', 'linewidth': 2})
for patch, col in zip(bp['boxes'], PALETTE_F):
    patch.set_facecolor(col); patch.set_alpha(0.7)
ax.set_xticks(range(1, k_best + 1))
ax.set_xticklabels([f'Cluster {c}\n(n={len(d)})' for c, d in enumerate(data_per_cluster)])
ax.set_ylabel('bAcc'); ax.set_title('bAcc per cluster')

# Mann-Whitney tra cluster 0 e 1
if k_best >= 2:
    d0 = data_per_cluster[0]; d1 = data_per_cluster[1]
    if len(d0) > 0 and len(d1) > 0:
        stat, pval = mannwhitneyu(d0, d1, alternative='two-sided')
        sig = '***' if pval < 0.001 else ('**' if pval < 0.01 else ('*' if pval < 0.05 else 'ns'))
        y_top = max(np.nanmax(d0), np.nanmax(d1)) + 0.01
        ax.plot([1, 2], [y_top, y_top], 'k-', lw=1)
        ax.text(1.5, y_top + 0.003, f'C0↔C1: p={pval:.3f} {sig}', ha='center', fontsize=9,
                color='dimgray')

# ── PCA 2D colorato per bAcc ─────────────────────────────────────────────────
ax = axes[1]
sc = ax.scatter(Z2[valid, 0], Z2[valid, 1],
                c=bacc_f[valid], cmap='RdYlGn', s=70,
                edgecolors=[PALETTE_F[l] for l in LABELS_F[valid]], linewidths=1.5)
plt.colorbar(sc, ax=ax, label='bAcc')
for i in range(len(SUBJ_F)):
    if valid[i]:
        ax.annotate(f'P{SUBJ_F[i]:03d}', (Z2[i, 0], Z2[i, 1]), fontsize=5.5,
                    ha='center', va='bottom', xytext=(0, 3), textcoords='offset points')
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})')
ax.set_title('PCA 2D — colore=bAcc, contorno=cluster')
# legenda cluster
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor='none',
                  markeredgecolor=PALETTE_F[c], markersize=9, markeredgewidth=2,
                  label=f'Cluster {c}') for c in range(k_best)]
ax.legend(handles=handles, fontsize=8)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(FIG_DIR / 'eeg18_cluster_vs_bacc.png', dpi=150, bbox_inches='tight')
plt.show()

## §7 — Quale Feature Funzionale Correla con bAcc?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Feature funzionali vs bAcc — correlazione per soggetto', fontsize=13, fontweight='bold')

labels_nice = ['Within-class consistency', 'Fisher ratio', 'Alpha power (8–12 Hz)', 'Trial variance']
colors_feat = ['#4878CF', '#6ACC65', '#E68310', '#D65F5F']

for ax, feat_col, name, col in zip(axes, FEAT_F.T, labels_nice, colors_feat):
    mask = valid & np.isfinite(feat_col)
    x_v  = feat_col[mask]
    y_v  = bacc_f[mask]
    r, p = pearsonr(x_v, y_v)
    sig  = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))

    ax.scatter(x_v, y_v, c=col, alpha=0.7, s=50, edgecolors='white', lw=0.5)

    # trend line
    coef = np.polyfit(x_v, y_v, 1)
    xline = np.linspace(x_v.min(), x_v.max(), 100)
    ax.plot(xline, np.polyval(coef, xline), 'k--', lw=1.5, alpha=0.6)

    ax.set_xlabel(name, fontsize=9)
    ax.set_ylabel('bAcc', fontsize=9)
    ax.set_title(f'r={r:.3f}  p={p:.3f} {sig}', fontsize=10, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(FIG_DIR / 'eeg18_feature_vs_bacc.png', dpi=150, bbox_inches='tight')
plt.show()

## §8 — Confronto con EEG_16: Funzionale vs Strutturale (ARI)

In [ ]:
# Carica cluster strutturali da EEG_16 summary CSV
eeg16_csv = project_root / 'figures' / 'eeg16_subject_cluster_summary.csv'

if eeg16_csv.exists():
    df16 = pd.read_csv(eeg16_csv)
    df16['sid'] = df16['Subject'].str.extract(r'(\d+)').astype(int)

    # allinea sui soggetti comuni
    sid2clust_struct = dict(zip(df16['sid'], df16['cluster_graph']))
    sid2clust_func   = dict(zip(SUBJ_F, LABELS_F))

    common = sorted(set(sid2clust_struct.keys()) & set(sid2clust_func.keys()))
    lbl_struct = np.array([sid2clust_struct[s] for s in common])
    lbl_func   = np.array([sid2clust_func[s]   for s in common])

    ari = adjusted_rand_score(lbl_struct, lbl_func)
    log.info(f'Soggetti comuni: {len(common)}  ARI strutturale vs funzionale = {ari:.4f}')

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Strutturale (EEG_16) vs Funzionale (EEG_18)', fontsize=13, fontweight='bold')

    # ── confusion-style heatmap ───────────────────────────────────────────────
    ax = axes[0]
    k_s = int(lbl_struct.max()) + 1
    k_f = k_best
    conf = np.zeros((k_s, k_f), dtype=int)
    for s, f in zip(lbl_struct, lbl_func):
        conf[int(s), int(f)] += 1
    im = ax.imshow(conf, cmap='YlOrRd', aspect='auto')
    plt.colorbar(im, ax=ax, label='n soggetti')
    ax.set_xticks(range(k_f)); ax.set_xticklabels([f'Func C{i}' for i in range(k_f)])
    ax.set_yticks(range(k_s)); ax.set_yticklabels([f'Struct C{i}' for i in range(k_s)])
    ax.set_xlabel('Cluster funzionale'); ax.set_ylabel('Cluster strutturale')
    ax.set_title(f'Sovrapposizione cluster\nARI = {ari:.3f}', fontsize=11)
    for i in range(k_s):
        for j in range(k_f):
            ax.text(j, i, str(conf[i, j]), ha='center', va='center',
                    fontsize=12, fontweight='bold',
                    color='white' if conf[i, j] > conf.max() * 0.6 else 'black')

    # ── scatter: cluster strutturale vs funzionale colorato per bAcc ──────────
    ax = axes[1]
    jitter = 0.12
    np.random.seed(42)
    bacc_common = np.array([ACC.get(s, np.nan) for s in common])
    valid_c = ~np.isnan(bacc_common)
    sc = ax.scatter(
        lbl_struct[valid_c] + np.random.uniform(-jitter, jitter, valid_c.sum()),
        lbl_func[valid_c]   + np.random.uniform(-jitter, jitter, valid_c.sum()),
        c=bacc_common[valid_c], cmap='RdYlGn', s=80,
        edgecolors='gray', linewidths=0.5, vmin=0.18, vmax=0.36)
    plt.colorbar(sc, ax=ax, label='bAcc')
    ax.set_xticks(range(k_s)); ax.set_xticklabels([f'Struct C{i}' for i in range(k_s)])
    ax.set_yticks(range(k_f)); ax.set_yticklabels([f'Func C{i}' for i in range(k_f)])
    ax.set_xlabel('Cluster strutturale (EEG_16)')
    ax.set_ylabel('Cluster funzionale (EEG_18)')
    ax.set_title(f'Soggetti colorati per bAcc\n(ARI={ari:.3f})', fontsize=11)

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(FIG_DIR / 'eeg18_ari_struct_vs_func.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    log.warning(f'EEG_16 CSV non trovato: {eeg16_csv}')
    log.warning('Esegui EEG_16 prima per generare eeg16_subject_cluster_summary.csv')

## §9 — Tabella Riepilogativa per Soggetto

In [ ]:
rows = []
for i, sid in enumerate(SUBJ_F):
    row = {'Subject': f'P{sid:03d}'}
    for j, fname in enumerate(FEAT_NAMES):
        row[fname] = float(FEAT_F[i, j])
    row['cluster_func'] = int(LABELS_F[i])
    row['bacc'] = ACC.get(sid, np.nan)
    rows.append(row)

df_summary = pd.DataFrame(rows)
df_summary.to_csv(FIG_DIR / 'eeg18_subject_functional_summary.csv', index=False)

# Statistiche per cluster
print('\n── Statistiche per cluster funzionale ──')
for c in range(k_best):
    sub = df_summary[df_summary['cluster_func'] == c]
    print(f'\nCluster {c}  (n={len(sub)}):')
    for fname in FEAT_NAMES:
        vals = sub[fname].dropna()
        print(f'  {fname:<25s}  mean={vals.mean():.4f}  std={vals.std():.4f}')
    bacc_c = sub['bacc'].dropna()
    print(f'  {"bAcc":<25s}  mean={bacc_c.mean():.4f}  std={bacc_c.std():.4f}')

display(df_summary.style.background_gradient(subset=['bacc'], cmap='RdYlGn').format(precision=4))

## §10 — Feature Set Esteso: Multi-Band Power + Li Gamma-Temporal

Estensione del clustering con feature più ricche:
- **A** — Multi-band power (δ/θ/α/β/γ × 61 el. = 305D)
- **B** — Li et al. 2025 12 temporal features gamma (× 61 el. = 732D)
- **C** — A + B + 4 scalari funzionali (1041D)


In [ ]:
from scipy.signal import butter, filtfilt
from scipy.signal import welch as sp_welch

BANDS = {'delta':(1,4), 'theta':(4,8), 'alpha':(8,12), 'beta':(13,30), 'gamma':(30,70)}
BAND_NAMES = list(BANDS.keys())

def _bandpass(lo, hi, fs=FS, order=4):
    nyq = fs / 2.0
    b, a = butter(order, [lo/nyq, hi/nyq], btype='bandpass')
    return b, a

BAND_FILTERS = {name: _bandpass(*fq) for name, fq in BANDS.items()}

def _entropy(sig, n_bins=16):
    p, _ = np.histogram(sig, bins=n_bins, density=True)
    p = p[p > 0]; p /= p.sum()
    return -np.sum(p * np.log2(p + 1e-12))

def temporal_features_12(sig):
    """sig: (T,) gamma-filtered -> (12,) float64"""
    T = len(sig); t = np.arange(T, dtype=np.float64)
    d = np.diff(sig); ad = np.abs(d)
    return np.array([
        np.dot(t, np.abs(sig)) / (np.abs(sig).sum() + 1e-12),
        ad.mean(), d.mean(), np.median(ad), np.median(d),
        np.sqrt(1.0 + d**2).sum(), ad.sum(),
        np.polyfit(t, sig, 1)[0],
        np.abs(sig).sum() / FS,
        sig.max() - sig.min(),
        _entropy(sig),
        float(((sig[1:-1]>sig[:-2])&(sig[1:-1]>sig[2:])).sum()),
    ], dtype=np.float64)

CACHE_10 = CKPT_DIR / 'eeg18_extended_feats.npz'

if CACHE_10.exists():
    log.info(f'Cache §10 trovata: {CACHE_10}')
    _c = np.load(CACHE_10)
    SUBJ_EXT  = _c['subj_ids'].tolist()
    FEAT_BAND = _c['feat_band']   # (n_subj, 305)
    FEAT_LI12 = _c['feat_li12']   # (n_subj, 732)
else:
    log.info('Calcolo feature estese per soggetto (~5-10 min)...')
    _b_g, _a_g = BAND_FILTERS['gamma']
    rows_band, rows_li12, subj_ids = [], [], []

    for sid in tqdm(ALL_SUBJ, desc='Soggetti'):
        paths = IDX[sid]
        band_acc, li12_acc = [], []
        for p in paths:
            try:
                d = torch.load(p, weights_only=False)
                x = d['x'].float().numpy()
                if x.shape != (N_CHAN, N_SAMPLES): continue
                # Multi-band power: media PSD per banda per canale
                bp = np.zeros((len(BAND_NAMES), N_CHAN), dtype=np.float64)
                for bi, bname in enumerate(BAND_NAMES):
                    lo, hi = BANDS[bname]
                    for ch in range(N_CHAN):
                        f_ax, psd = sp_welch(x[ch], fs=FS, nperseg=min(128, N_SAMPLES))
                        mask_b = (f_ax >= lo) & (f_ax < hi)
                        bp[bi, ch] = psd[mask_b].mean() if mask_b.any() else 0.0
                band_acc.append(bp)
                # Li 12 temporal su gamma
                xg = filtfilt(_b_g, _a_g, x, axis=1)
                li12 = np.stack([temporal_features_12(xg[ch]) for ch in range(N_CHAN)])
                li12_acc.append(li12)
            except Exception:
                continue

        if len(band_acc) < 20:
            log.warning(f'P{sid:03d}: {len(band_acc)} trial — skip')
            continue

        rows_band.append(np.stack(band_acc).mean(axis=0).ravel())   # (305,)
        rows_li12.append(np.stack(li12_acc).mean(axis=0).ravel())   # (732,)
        subj_ids.append(sid)

    SUBJ_EXT  = subj_ids
    FEAT_BAND = np.array(rows_band, dtype=np.float32)
    FEAT_LI12 = np.array(rows_li12, dtype=np.float32)
    np.savez(CACHE_10, subj_ids=np.array(SUBJ_EXT),
             feat_band=FEAT_BAND, feat_li12=FEAT_LI12)
    log.info(f'Salvato: {CACHE_10} — {len(SUBJ_EXT)} soggetti')

log.info(f'FEAT_BAND: {FEAT_BAND.shape}  FEAT_LI12: {FEAT_LI12.shape}')


In [ ]:
# Allinea con SUBJ_F per aggiungere i 4 scalari
common_subj = sorted(set(SUBJ_EXT) & set(SUBJ_F))
idx_ext = [SUBJ_EXT.index(s) for s in common_subj]
idx_f   = [SUBJ_F.index(s)   for s in common_subj]

F_band = FEAT_BAND[idx_ext]
F_li12 = FEAT_LI12[idx_ext]
F_scal = FEAT_F[idx_f]
F_all  = np.hstack([F_band, F_li12, F_scal])

n_common = len(common_subj)
log.info(f'Soggetti comuni: {n_common}')

FEATURE_SETS = {
    'A — Multi-band (305D)':        F_band,
    'B — Li Gamma-Temporal (732D)': F_li12,
    'C — A+B+4scalari (1041D)':     F_all,
}

fig, axes = plt.subplots(len(FEATURE_SETS), len(K_RANGE)+1,
                          figsize=(5*(len(K_RANGE)+1), 4*len(FEATURE_SETS)))
fig.suptitle('EEG_18 §10 — Clustering con feature estese', fontsize=14, fontweight='bold')

results_ext = {}

for row_i, (fs_name, F_raw) in enumerate(FEATURE_SETS.items()):
    F_sc = StandardScaler().fit_transform(np.nan_to_num(F_raw, nan=0.0))
    n_comp_i = min(n_common - 1, F_sc.shape[1], 20)
    pca_i = PCA(n_components=n_comp_i, random_state=RANDOM_SEED)
    F_pca = pca_i.fit_transform(F_sc)
    var_exp = pca_i.explained_variance_ratio_.sum()

    sil_scores = []
    best_k, best_sil, best_labels = K_RANGE[0], -1, None

    for col_i, k in enumerate(K_RANGE):
        km  = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=20)
        lbl = km.fit_predict(F_pca)
        sil = silhouette_score(F_pca, lbl) if len(np.unique(lbl)) > 1 else 0.0
        sil_scores.append(sil)
        if sil > best_sil:
            best_k, best_sil, best_labels = k, sil, lbl

        ax = axes[row_i, col_i]
        ax.scatter(F_pca[:, 0], F_pca[:, 1], c=lbl, cmap='tab10', s=30, alpha=0.8)
        ax.set_title(f'{fs_name}\nk={k} sil={sil:.3f}', fontsize=9)
        ax.set_xlabel(f'PC1 ({pca_i.explained_variance_ratio_[0]:.1%})', fontsize=8)
        ax.set_ylabel(f'PC2 ({pca_i.explained_variance_ratio_[1]:.1%})', fontsize=8)

    ax_sil = axes[row_i, -1]
    ax_sil.bar(K_RANGE, sil_scores,
               color=['#2C7BB6' if s == max(sil_scores) else '#AAAAAA' for s in sil_scores])
    ax_sil.set_title(f'{fs_name}\nSilhouette per k  (var={var_exp:.0%})', fontsize=9)
    ax_sil.set_xlabel('k'); ax_sil.set_ylabel('Silhouette'); ax_sil.set_ylim(0, 1)
    ax_sil.set_xticks(K_RANGE)
    results_ext[fs_name] = dict(best_k=best_k, best_sil=best_sil,
                                 best_labels=best_labels, var_exp=var_exp, F_pca=F_pca)

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg18_ext_clustering.png', dpi=150); plt.show()
log.info(f'Salvato: {FIG_DIR}/eeg18_ext_clustering.png')


In [ ]:
from scipy.stats import kruskal, spearmanr

bacc_common = np.array([ACC.get(s, np.nan) for s in common_subj])
valid_bacc  = ~np.isnan(bacc_common)

fig2, axes2 = plt.subplots(1, len(FEATURE_SETS), figsize=(6*len(FEATURE_SETS), 5))
fig2.suptitle('EEG_18 §10 — Cluster vs bAcc (best k)', fontsize=13, fontweight='bold')

print("\n── Validazione cluster vs bAcc ──")
print(f"{'Feature set':<35} {'best_k':>6} {'sil':>6} {'KW p':>8} {'Sp r':>8}")
print('-'*67)

for ax_i, (fs_name, res) in enumerate(results_ext.items()):
    ax = axes2[ax_i]
    lbl = res['best_labels']; k = res['best_k']
    groups = [bacc_common[valid_bacc & (lbl == c)] for c in range(k) if (valid_bacc & (lbl==c)).sum() > 1]
    kw_p = kruskal(*groups).pvalue if len(groups) >= 2 else np.nan
    sp_r, _ = spearmanr(lbl[valid_bacc], bacc_common[valid_bacc])
    bp_data = [bacc_common[(lbl == c) & valid_bacc] for c in range(k)]
    ax.boxplot(bp_data, labels=[f'C{c}\n(n={len(d)})' for c, d in enumerate(bp_data)])
    ax.axhline(0.25, color='k', linestyle='--', linewidth=1, label='chance')
    ax.set_title(f'{fs_name}\nk={k}  sil={res["best_sil"]:.3f}\nKW p={kw_p:.3f}  Sp r={sp_r:.3f}', fontsize=9)
    ax.set_ylabel('bAcc'); ax.legend(fontsize=8)
    print(f"{fs_name:<35} {k:>6} {res['best_sil']:>6.3f} {kw_p:>8.3f} {sp_r:>8.3f}")

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg18_ext_cluster_vs_bacc.png', dpi=150); plt.show()

# ARI vs strutturale EEG_16
print("\n── ARI vs clustering strutturale EEG_16 ──")
eeg16_csv = project_root / 'figures' / 'eeg16_subject_cluster_summary.csv'
if eeg16_csv.exists():
    df16 = pd.read_csv(eeg16_csv)
    struct_map = {}
    for _, row16 in df16.iterrows():
        sid_s = str(row16.get('subject_id', row16.get('Subject', '')))
        sid_i = int(re.sub(r"[^0-9]", "", sid_s)) if re.search(r"[0-9]", sid_s) else None
        if sid_i is not None:
            struct_map[sid_i] = int(row16.get('cluster', row16.get('Cluster', 0)))
    for fs_name, res in results_ext.items():
        pairs = [(s, res['best_labels'][i]) for i, s in enumerate(common_subj) if s in struct_map]
        if len(pairs) < 10: continue
        sids2, func_lbl = zip(*pairs)
        struct_lbl = [struct_map[s] for s in sids2]
        ari = adjusted_rand_score(struct_lbl, func_lbl)
        print(f"  {fs_name:<35}: ARI = {ari:.4f}")
else:
    print("  EEG_16 CSV non trovato — skip ARI")
